In [2]:
%run C:\Users\gabri\Documents\Chicago\Palm_watch\PalmWatch\ImportFilesAndFunctions.ipynb

Seed set to 42


All imports successful!


In [3]:
# load the dataset
palmoil_df = csv_old

# check for any null values
palmoil_df[palmoil_df.isnull().any(axis=1)]

# fills all empty/na/nan values (usually provinces with no neighbors) with 0
palmoil_df.fillna(0, inplace=True)

In [4]:
# Get data from 2000 up until 2018
palmoil_wo_target = palmoil_df[palmoil_df['year'] <= 2018]

In [5]:
feature_names = ['nbr_r1_defor_frac', 'remaining_frac_hex', 'remaining_forest_ha', 'remaining_frac_baseline', 'defor_frac', 'nbr_r1_remaining_frac_baseline' , 'nbr_r2_defor_frac', 'degr_frac', 'target_total_5yr', 'target_palm_5yr']

In [6]:
palmoil_no_target = palmoil_wo_target.drop(columns = ["target_total_5yr", "target_palm_5yr"])
palmoil_no_target

,hex_id,year,province,lat,lon,hex_area_km2,forest_2000_ha,forest_2000_frac,defor_frac,degr_frac,remaining_frac_baseline,remaining_frac_hex,remaining_forest_ha,nbr_r1_defor_frac,nbr_r2_defor_frac,nbr_r1_remaining_frac_baseline
0,86642c96fffffff,2000,Aceh,5.864320,95.333609,43.2561,2009.9502,0.464663,0.000000,0.000000,1.000000,0.464663,2009.95,0.000000,0.000000,1.000000
1,86642c96fffffff,2001,Aceh,5.864320,95.333609,43.2561,2009.9502,0.464663,0.032801,0.027606,0.967199,0.449421,1944.02,0.006728,0.000000,0.993272
2,86642c96fffffff,2002,Aceh,5.864320,95.333609,43.2561,2009.9502,0.464663,0.021762,0.024175,0.945437,0.439309,1900.28,0.008174,0.000000,0.985098
3,86642c96fffffff,2003,Aceh,5.864320,95.333609,43.2561,2009.9502,0.464663,0.001903,0.000973,0.943534,0.438425,1896.46,0.000895,0.000000,0.984203
4,86642c96fffffff,2004,Aceh,5.864320,95.333609,43.2561,2009.9502,0.464663,0.011296,0.007714,0.932238,0.433177,1873.75,0.004071,0.000000,0.980132
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
208982,868cf6db7ffffff,2014,SumateraSelatan,-3.585241,104.466315,38.8215,1365.4811,0.351733,0.025313,0.006765,0.504889,0.177586,689.42,0.016244,0.019381,0.636175
208983,868cf6db7ffffff,2015,SumateraSelatan,-3.585241,104.466315,38.8215,1365.4811,0.351733,0.011539,0.005554,0.493351,0.173528,673.66,0.018627,0.019631,0.617548
208984,868cf6db7ffffff,2016,SumateraSelatan,-3.585241,104.466315,38.8215,1365.4811,0.351733,0.013483,0.004640,0.479867,0.168785,655.25,0.021773,0.013013,0.595775
208985,868cf6db7ffffff,2017,SumateraSelatan,-3.585241,104.466315,38.8215,1365.4811,0.351733,0.005961,0.002745,0.473906,0.166689,647.11,0.004125,0.004524,0.591649


In [7]:
# Check how many hex ids each province contains
hex_per_province = palmoil_wo_target.groupby("province")["hex_id"].nunique().reset_index()
hex_per_province

,province,hex_id
0,Aceh,1223
1,Bengkulu,405
2,Jambi,1135
3,Lampung,238
4,Riau,1745
5,SumateraBarat,966
6,SumateraSelatan,1686
7,SumateraUtara,1310


In [8]:
# Check if the count for each province is correct
total_hexs = hex_per_province["hex_id"].sum()
total_hexs

np.int64(8708)

In [9]:
# Creates a dataframe for a specific province and year
# The dataframe contains all of the features including lat and lon values
def sortByProvince(province, df, year):
  """Create a new dataframe and return all rows of a specific province and specific year"""
  new_df = df[(df['province'] == province) & (df['year'] == year)]

  return new_df

In [10]:
# New dataframe grouped by 16 hex ids inside each province
def create_subsetted_data(province, df, year, feature_names, group_size=16):
  """Creates a new dataframe that finds the median of a group of 16 hex_id"""

  # Dataframe with all features
  province_df = sortByProvince(province, df, year)

  # Dictionary to create new subset
  subsetted_data = {
      "Grouped ID": [],
      "Year": []
  }

  # Add a column for each feature to calculate its median value
  for feature in feature_names:
      subsetted_data[f"{feature}_median"] = []

  # Add lat and lon columns
  subsetted_data["lat"] = []
  subsetted_data["lon"] = []
  subsetted_data["hex_area_16"] = []

  # Iterate through the whole dataframe with a step of 16
  # to create new groups with 16 hex ids each
  for i in range(0, len(province_df), group_size):
      subsetted_data["Grouped ID"].append(i // group_size)
      subsetted_data["Year"].append(year)

      # Remove the index of the series by converting the data to a numpy array
      latitudes_16hex = (province_df["lat"].iloc[i:i+group_size]).to_numpy()
      longitudes_16hex = (province_df["lon"].iloc[i:i+group_size]).to_numpy()
      hex_area_16 = (province_df["hex_area_km2"].iloc[i:i+group_size]).to_numpy()

      # Add all hex areas to calculate total for these new subdivions
      shex_area_16 = sum(hex_area_16)

      # Save lat and lon data
      subsetted_data["lat"].append(tuple(latitudes_16hex))
      subsetted_data["lon"].append(tuple(longitudes_16hex))
      subsetted_data["hex_area_16"].append(shex_area_16)
      #print(f"Lat len: {len(tuple(latitudes_16hex))}\n Lon len: {len(tuple(longitudes_16hex))}")

      # Calculate median for each feature and save it in the subsetted data dict
      for feature in feature_names:
          subsetted_data[f"{feature}_median"].append(
              province_df[feature].iloc[i:i+group_size].median())

  return pd.DataFrame(subsetted_data)

In [11]:
def create_all_years_data(province, df, years, feature_names, group_size=16):
  """Creates a dataframe with all years of a specific province"""
  all_data = []
  df = df.sort_values(['year', 'province', 'lat', 'lon'])

  for year in years:
      yearly_df = create_subsetted_data(
          province,
          df,
          year,
          feature_names,
          group_size
      )

      all_data.append(yearly_df)

  return pd.concat(all_data, ignore_index=True)

In [12]:
YEARS = range(2000, 2019)

In [13]:
combined_aceh = create_all_years_data(
    province="Aceh",
    df=palmoil_wo_target,
    years=YEARS,
    feature_names=feature_names)

combined_aceh.insert(2, 'Province', 'Aceh')

combined_sumaterautara = create_all_years_data(
    province="SumateraUtara",
    df=palmoil_wo_target,
    years=YEARS,
    feature_names=feature_names
)

combined_sumaterautara.insert(2, 'Province', 'SumateraUtara')

combined_riau = create_all_years_data(
    province="Riau",
    df=palmoil_wo_target,
    years=YEARS,
    feature_names=feature_names
)

combined_riau.insert(2, 'Province', 'Riau')

combined_sumaterabarat = create_all_years_data(
    province="SumateraBarat",
    df=palmoil_wo_target,
    years=YEARS,
    feature_names=feature_names
)

combined_sumaterabarat.insert(2, 'Province', 'SumateraBarat')

combined_lampung = create_all_years_data(
    province="Lampung",
    df=palmoil_wo_target,
    years=YEARS,
    feature_names=feature_names
)

combined_lampung.insert(2, 'Province', 'Lampung')

combined_sumateraselatan = create_all_years_data(
    province="SumateraSelatan",
    df=palmoil_wo_target,
    years=YEARS,
    feature_names=feature_names
)

combined_sumateraselatan.insert(2, 'Province', 'SumateraSelatan')

combined_jambi = create_all_years_data(
    province="Jambi",
    df=palmoil_wo_target,
    years=YEARS,
    feature_names=feature_names
)

combined_jambi.insert(2, 'Province', 'Jambi')

combined_bengkulu = create_all_years_data(
    province="Bengkulu",
    df=palmoil_wo_target,
    years=YEARS,
    feature_names=feature_names
)

combined_bengkulu.insert(2, 'Province', 'Bengkulu')

combined_aceh

,Grouped ID,Year,Province,nbr_r1_defor_frac_median,remaining_frac_hex_median,remaining_forest_ha_median,remaining_frac_baseline_median,defor_frac_median,nbr_r1_remaining_frac_baseline_median,nbr_r2_defor_frac_median,degr_frac_median,target_total_5yr_median,target_palm_5yr_median,lat,lon,hex_area_16
0,0,2000,Aceh,0.000000,0.581891,2466.300,1.000000,0.000000,1.000000,0.000000,0.000000,0.016262,0.000000,"(2.0602378096361584, 2.093238789009517, 2.0977...","(97.118100450005, 96.67672251895677, 97.343008...",678.1160
1,1,2000,Aceh,0.000000,0.733730,3112.350,1.000000,0.000000,1.000000,0.000000,0.000000,0.065601,0.003927,"(2.313553289379441, 2.327863911042388, 2.32851...","(98.08962112312604, 97.90052866862926, 96.4410...",678.8557
2,2,2000,Aceh,0.000000,0.720831,3060.465,1.000000,0.000000,1.000000,0.000000,0.000000,0.039920,0.003172,"(2.4217904384204147, 2.426569715962156, 2.4313...","(98.098452336715, 98.0354078956144, 97.9723570...",679.2776
3,3,2000,Aceh,0.000000,0.911251,3869.955,1.000000,0.000000,1.000000,0.000000,0.000000,0.016555,0.000000,"(2.498065261222968, 2.5022014076727226, 2.5027...","(96.35874696018963, 97.75599425606732, 96.2955...",679.8743
4,4,2000,Aceh,0.000000,0.791338,3366.975,1.000000,0.000000,1.000000,0.000000,0.000000,0.030527,0.000000,"(2.5734046602046647, 2.5827832296706763, 2.586...","(96.07863231165284, 95.9522863507064, 98.08017...",680.2275
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1458,72,2018,Aceh,0.002650,0.458662,1978.995,0.838728,0.002302,0.859873,0.001570,0.006812,0.027654,0.000000,"(5.183553097357718, 5.183580950605123, 5.18826...","(97.07657903888496, 95.60396404573336, 95.5402...",690.3586
1459,73,2018,Aceh,0.000741,0.878522,3791.280,0.985380,0.000186,0.968813,0.000971,0.001308,0.001696,0.000000,"(5.217159438837113, 5.2218721129808365, 5.2265...","(95.89458052431566, 95.8308999439684, 95.76721...",690.5298
1460,74,2018,Aceh,0.000898,0.497341,2146.790,0.956668,0.000519,0.958876,0.001061,0.003040,0.013314,0.000000,"(5.288368616980756, 5.29774362782648, 5.302421...","(95.67575296027525, 95.5483643149806, 95.48466...",690.7514
1461,75,2018,Aceh,0.001422,0.492933,2129.735,0.919992,0.001132,0.924013,0.001300,0.003097,0.027081,0.000000,"(5.39318107641495, 5.397879728895629, 5.407257...","(95.74758884087927, 95.68387430955752, 95.5564...",691.0761


In [14]:
overall_grouped = pd.concat([combined_aceh, combined_sumaterautara, combined_riau, combined_sumaterabarat, combined_lampung, combined_sumateraselatan, combined_jambi, combined_bengkulu])

print('New Dataframe Shape: ', overall_grouped.shape)
rows, columns = overall_grouped.shape

New Dataframe Shape:  (10412, 16)


In [15]:
overall_grouped.tail(10)

,Grouped ID,Year,Province,nbr_r1_defor_frac_median,remaining_frac_hex_median,remaining_forest_ha_median,remaining_frac_baseline_median,defor_frac_median,nbr_r1_remaining_frac_baseline_median,nbr_r2_defor_frac_median,degr_frac_median,target_total_5yr_median,target_palm_5yr_median,lat,lon,hex_area_16
484,16,2018,Bengkulu,0.009361,0.321090,1274.775,0.731451,0.010024,0.750416,0.007298,0.004523,0.047105,0.0,"(-3.1874281522014143, -3.183296059119031, -3.1...","(102.39007911447452, 102.32882437819416, 102.2...",634.2094
485,17,2018,Bengkulu,0.008620,0.730707,2899.605,0.867262,0.005439,0.799397,0.007140,0.004498,0.023303,0.0,"(-3.1250678521082795, -3.1209165472757547, -3....","(102.24187478511304, 102.18057187695447, 102.1...",634.9033
486,18,2018,Bengkulu,0.002279,0.973942,3871.135,0.990445,0.000412,0.933465,0.006592,0.002450,0.003659,0.0,"(-3.066785882334583, -3.062620287316805, -3.05...","(102.15486530816882, 102.0935303740952, 102.03...",635.8188
487,19,2018,Bengkulu,0.002615,0.820549,3264.360,0.901556,0.003207,0.865341,0.008016,0.001553,0.026392,0.0,"(-2.9833049986439253, -2.979098418729972, -2.9...","(101.69937074879763, 101.6379148718692, 101.57...",636.8879
488,20,2018,Bengkulu,0.000517,0.993011,3953.795,0.998892,0.000023,0.989969,0.000620,0.000203,0.000590,0.0,"(-2.908407484862276, -2.9042176942690534, -2.9...","(102.1390479488452, 102.07767912283522, 102.01...",637.2258
489,21,2018,Bengkulu,0.000021,0.993856,3964.090,0.999521,0.000022,0.998969,0.002038,0.000147,0.000167,0.0,"(-2.833129440627443, -2.828908544292666, -2.82...","(101.80623538202876, 101.74477489343832, 101.6...",638.3931
490,22,2018,Bengkulu,0.004046,0.986637,3938.430,0.994676,0.000186,0.948750,0.005769,0.000935,0.001604,0.0,"(-2.761870097327036, -2.757623244799533, -2.75...","(101.53441470214868, 101.47288053342784, 101.4...",639.1556
491,23,2018,Bengkulu,0.001466,0.994802,3977.765,0.999788,0.000000,0.960633,0.004624,0.000135,0.000045,0.0,"(-2.670101650942481, -2.665853517153748, -2.66...","(101.72872103693297, 101.6672121129823, 101.60...",640.3422
492,24,2018,Bengkulu,0.012358,0.656678,2633.855,0.805309,0.009460,0.766099,0.007424,0.004136,0.115885,0.0,"(-2.5441859728898417, -2.539904129553873, -2.5...","(101.43073962918815, 101.36914182950412, 101.3...",641.9277
493,25,2018,Bengkulu,0.004129,0.994669,3997.440,0.999821,0.000000,0.911994,0.007811,0.000067,0.001366,0.0,"(-2.380627974110087, -2.3763204474754445, -2.3...","(101.35286868652356, 101.29122334579678, 101.2...",200.9380


In [16]:
# Group dataset by year to map the new subsets
deforest_hex = overall_grouped.groupby(["Grouped ID", "Province", "lat", "lon", "hex_area_16"])[["remaining_frac_baseline_median", "defor_frac_median"]].median().reset_index()
deforest_hex

,Grouped ID,Province,lat,lon,hex_area_16,remaining_frac_baseline_median,defor_frac_median
0,0,Aceh,"(2.0602378096361584, 2.093238789009517, 2.0977...","(97.118100450005, 96.67672251895677, 97.343008...",678.1160,0.970606,0.000942
1,0,Bengkulu,"(-5.41217392335722, -5.3592568889426175, -5.35...","(102.3293359394739, 102.3038516742851, 102.242...",615.4465,0.893346,0.007208
2,0,Jambi,"(-2.745605567343052, -2.699722822724129, -2.69...","(102.06181138042396, 102.15886663922932, 102.0...",637.9543,0.999285,0.000043
3,0,Lampung,"(-6.15612867130734, -5.91063978687678, -5.9071...","(105.43969693751674, 104.68205994141807, 104.6...",599.7768,0.988654,0.000323
4,0,Riau,"(-1.0961512190299951, -1.0917140922937305, -1....","(102.65220757888136, 102.59062232523804, 102.6...",647.9604,0.999955,0.000000
...,...,...,...,...,...,...,...
543,105,SumateraSelatan,"(-1.7436373022043763, -1.7308236377463395, -1....","(104.31646440340576, 104.13345778684916, 104.4...",238.3479,0.989555,0.001040
544,106,Riau,"(2.054897019329136, 2.0597466134476083, 2.0645...","(100.75542472372976, 100.6928172654304, 100.63...",671.4144,0.775986,0.016021
545,107,Riau,"(2.120929378393521, 2.125781110382992, 2.13063...","(100.60368267225326, 100.54103447619748, 100.4...",672.1405,0.742644,0.013180
546,108,Riau,"(2.199573924135511, 2.20444762962743, 2.209319...","(100.9890224948335, 100.92643256291552, 100.86...",672.8457,0.791951,0.024938


In [17]:
# Separate lat and lon into different rows
# explode divides a tuple and disperses it throughout the rows keeping the same original values of the other columns
deforest_hex = deforest_hex.explode(column = ["lat", "lon"])
deforest_hex

,Grouped ID,Province,lat,lon,hex_area_16,remaining_frac_baseline_median,defor_frac_median
0,0,Aceh,2.060238,97.1181,678.1160,0.970606,0.000942
0,0,Aceh,2.093239,96.676723,678.1160,0.970606,0.000942
0,0,Aceh,2.097761,97.343008,678.1160,0.970606,0.000942
0,0,Aceh,2.097944,96.613655,678.1160,0.970606,0.000942
0,0,Aceh,2.149682,96.649369,678.1160,0.970606,0.000942
...,...,...,...,...,...,...,...
546,108,Riau,2.304641,100.335953,672.8457,0.791951,0.024938
546,108,Riau,2.356196,100.372082,672.8457,0.791951,0.024938
546,108,Riau,2.361064,100.309348,672.8457,0.791951,0.024938
546,108,Riau,2.412633,100.345482,672.8457,0.791951,0.024938


In [18]:
deforest_hex.info()

<class 'pandas.core.frame.DataFrame'>
Index: 8708 entries, 0 to 547
Data columns (total 7 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   Grouped ID                      8708 non-null   int64  
 1   Province                        8708 non-null   object 
 2   lat                             8708 non-null   object 
 3   lon                             8708 non-null   object 
 4   hex_area_16                     8708 non-null   float64
 5   remaining_frac_baseline_median  8708 non-null   float64
 6   defor_frac_median               8708 non-null   float64
dtypes: float64(3), int64(1), object(3)
memory usage: 802.3+ KB


**Mapping Subset data**

In [19]:
# Plotly to map new data subset regions
import plotly.express as px

# Figure out if size and shape are affecting the way the data is visualized
# Create the map trace
fig = px.scatter_map(
    deforest_hex,
    lat= "lat",
    lon="lon",
    hover_name="Grouped ID",
    hover_data = ["remaining_frac_baseline_median"],
    size="hex_area_16",
    color="Province",
    zoom=5,
    map_style = "satellite",
    title = "Remaining Forest Fraction of Hexagons in Sumatra Subset",
    width = 1000,
    height = 1000
)

fig.show()

**Mapping by Province**

In [20]:
# List of provinces
provinces = ["Jambi", "Aceh", "Riau", "Bengkulu", "Lampung", "SumateraSelatan", "SumateraUtara", "SumateraBarat"]

# Create a dataframe for each province
province_df = {}

for p in provinces:
  province_df[p] = deforest_hex[deforest_hex["Province"] == p]

# Features to plot in scatter map
features = ["remaining_frac_baseline_median", "defor_frac_median"]

In [41]:
# Coloring by grouped ID
for key in province_df:

    df = province_df[key]
    title = f"Grouped IDs in {key}, Sumatra Subset data"

    df["Grouped ID"] = df["Grouped ID"].astype(str)

    # Create the map trace
    fig = px.scatter_map(
        df,
        lat= "lat",
        lon="lon",
        hover_name="Province",
        hover_data = ["Grouped ID", "hex_area_16"],
        size="hex_area_16",
        color= "Grouped ID",
        color_discrete_sequence=px.colors.qualitative.Light24,
        zoom=5,
        map_style = "satellite",
        title = title,
        width = 1000,
        height = 1000)

    fig.show()

/tmp/ipykernel_522/3105123434.py:7: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy



/tmp/ipykernel_522/3105123434.py:7: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy



/tmp/ipykernel_522/3105123434.py:7: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy



/tmp/ipykernel_522/3105123434.py:7: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy



/tmp/ipykernel_522/3105123434.py:7: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy



/tmp/ipykernel_522/3105123434.py:7: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy



/tmp/ipykernel_522/3105123434.py:7: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy



/tmp/ipykernel_522/3105123434.py:7: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy



In [22]:
# Iterate through the dictionary and create a map for each province

for f in features:
  for key in province_df:

    df = province_df[key]
    title = f"{f} in {key}, Sumatra Subset data"

    # Figure out if size and shape are affecting the way the data is visualized
    # Create the map trace
    fig = px.scatter_map(
        df,
        lat= "lat",
        lon="lon",
        hover_name="Province",
        hover_data = ["Grouped ID", "hex_area_16"],
        size="hex_area_16",
        color=f,
        color_continuous_scale= "Cividis",
        zoom=5,
        map_style = "satellite",
        title = title,
        width = 1000,
        height = 1000)

    fig.show()